# Make 2DAlphabet ROOT inputs

Reads coffea output files and writes `mtt_vs_mt` histograms into ROOT files
suitable for the 2DAlphabet fit framework, split by pass/fail and central/forward
analysis categories. Writes data, TTbar, and one signal ROOT file for the
nominal 2024 first-pass fit.


In [130]:
import sys
from pathlib import Path
import uproot
from coffea.util import load

cwd = Path.cwd().resolve()
REPO_ROOT = next(
    p for p in [cwd, *cwd.parents]
    if (p / "python").is_dir() and (p / "outputs").is_dir()
)
if str(REPO_ROOT / "python") not in sys.path:
    sys.path.append(str(REPO_ROOT / "python"))

year = "2024"
coffea_dir = REPO_ROOT / "outputs" / "dy"
out_dir = REPO_ROOT / "outputs" / "2dalphabet_inputs"

## Shared helpers

In [131]:
# at = >=1 top-tagged jet; 2t = >=2; cen/fwd = barrel/forward
ANCAT = {"atcen": 0, "atfwd": 1, "2tcen": 2, "2tfwd": 3}

# histogram key names expected by 2DAlphabet
HIST_KEYS = {
    "pass_cen": f"MttvsMtCen{year}Pass",
    "fail_cen": f"MttvsMtCen{year}Fail",
    "pass_fwd": f"MttvsMtFwd{year}Pass",
    "fail_fwd": f"MttvsMtFwd{year}Fail",
}

sel = {"systematic": "nominal"}

def split_pass_fail(h):
    return {
        "pass_cen": h[{**sel, "anacat": ANCAT["2tcen"]}],
        "fail_cen": h[{**sel, "anacat": ANCAT["atcen"]}],
        "pass_fwd": h[{**sel, "anacat": ANCAT["2tfwd"]}],
        "fail_fwd": h[{**sel, "anacat": ANCAT["atfwd"]}],
    }

def write_2dalphabet_root(out_path, h):
    pieces = split_pass_fail(h)
    with uproot.recreate(out_path) as f:
        for key, hist_name in HIST_KEYS.items():
            f[hist_name] = pieces[key]
    print(f"Wrote {out_path}")


## Data

In [132]:
data_paths = sorted(coffea_dir.glob(f"data_{year}_*_noSyst.coffea"))

data_total = None
for path in data_paths:
    h = load(path)["mtt_vs_mt"]
    data_total = h if data_total is None else data_total + h

In [133]:
data_hists = split_pass_fail(data_total)


In [134]:
data_out = out_dir / f"data_{year}.root"
write_2dalphabet_root(data_out, data_total)


Wrote /home/cms-jovyan/new_git/TTbarHadronicSkimmer_coffea2025/outputs/2dalphabet_inputs/data_2024.root


## TTbar (MC)

In [135]:
ttbar_paths = sorted(coffea_dir.glob(f"TTbar_{year}_inclusive_noSyst*.coffea"))
if not ttbar_paths:
    raise FileNotFoundError(f"No TTbar {year} noSyst coffea file found in {coffea_dir}")
h_ttbar = load(ttbar_paths[0])["mtt_vs_mt"]
print(f"Using TTbar input: {ttbar_paths[0]}")


In [136]:
ttbar_hists = split_pass_fail(h_ttbar)


In [137]:
ttbar_out = out_dir / f"TTbar_{year}.root"
write_2dalphabet_root(ttbar_out, h_ttbar)


Wrote /home/cms-jovyan/new_git/TTbarHadronicSkimmer_coffea2025/outputs/2dalphabet_inputs/TTbar_2024.root


## Signal (MC)


In [ ]:
signal_paths = sorted(
    p for p in coffea_dir.glob(f"ZPrime*{year}*noSyst*.coffea")
    if "_old" not in p.name
)
if not signal_paths:
    signal_paths = sorted(
        p for p in coffea_dir.glob(f"ZPrime*{year}*.coffea")
        if "_old" not in p.name
    )
if not signal_paths:
    raise FileNotFoundError(f"No ZPrime {year} coffea file found in {coffea_dir}")

signal_path = signal_paths[0]
h_signal = load(signal_path)["mtt_vs_mt"]
print(f"Using signal input: {signal_path}")


In [ ]:
signal_hists = split_pass_fail(h_signal)


In [ ]:
signal_out = out_dir / f"signal_{year}.root"
write_2dalphabet_root(signal_out, h_signal)


## Save histograms

In [138]:
import pickle

hists = {
    "data": data_total,
    "ttbar": h_ttbar,
    "signal": h_signal,
}

pkl_out = out_dir / f"hists_{year}.pkl"
with open(pkl_out, "wb") as f:
    pickle.dump(hists, f)

print(f"Wrote {pkl_out}")


Wrote /home/cms-jovyan/new_git/TTbarHadronicSkimmer_coffea2025/outputs/2dalphabet_inputs/hists_2024.pkl
